# 01 — Simulation du champ

Ce notebook illustre la **Phase 1** : génération d'un champ synthétique de
laitue de 100×1000 plants avec :
- une prévalence de base ~5 %,
- un effet de bordure exponentiel,
- une autocorrélation spatiale Matérn (portée ~10 m),
- 3 à 8 foyers d'infestation gaussiens 2D.

On visualise la probabilité vraie et la présence binaire, puis on calcule
quelques statistiques descriptives.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aphid_spatial.simulation import FieldConfig, simulate_field
from aphid_spatial.visualization.maps import plot_field

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")
FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
config = FieldConfig(seed=42)
field = simulate_field(config)
print(f"N plants : {field.coords.shape[0]:,}")
print(f"prévalence empirique : {field.presence.mean():.4f}")
print(f"min/max p : {field.prob.min():.4f} / {field.prob.max():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5), constrained_layout=True)
plot_field(field, "prob", ax=axes[0], title="Probabilité vraie p(x, y)")
plot_field(field, "presence", ax=axes[1], cmap="Greys", title="Présence Bernoulli y")
fig.savefig(FIG_DIR / "01_simulation_field.png", dpi=150)
plt.show()

## Statistiques descriptives par bande de distance au bord

In [ ]:
cfg = field.config
rows = np.arange(cfg.n_rows)
cols = np.arange(cfg.n_cols)
dy = np.minimum(rows, cfg.n_rows - 1 - rows) * cfg.spacing_m
dx = np.minimum(cols, cfg.n_cols - 1 - cols) * cfg.spacing_m
dist = np.minimum(dy[:, None], dx[None, :]).ravel()

edges = [0, 5, 10, 20, 30, 100]
for i in range(len(edges) - 1):
    lo, hi = edges[i], edges[i + 1]
    mask = (dist >= lo) & (dist < hi)
    if mask.any():
        print(f"d ∈ [{lo:>3}, {hi:>3}) m : n={mask.sum():>6}, p̄={field.prob[mask].mean():.4f}, ȳ={field.presence[mask].mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(field.prob, bins=50, edgecolor="black", alpha=0.7)
ax.set_xlabel("p (probabilité vraie)")
ax.set_ylabel("nombre de plants")
ax.set_title("Distribution de p sur le champ entier")
fig.savefig(FIG_DIR / "01_simulation_phist.png", dpi=150)
plt.show()

## Sauvegarde

On stocke le champ sur disque pour le réutiliser dans les notebooks suivants.

In [ ]:
data_path = Path("../data/simulated/field_default.npz")
field.save(data_path)
print(f"Champ sauvegardé : {data_path}")